# 🏏 CricAnalytica: ICC Men's T20 World Cup 2026 — Data Preprocessing & Insights
**Tournament:** ICC Men's T20 World Cup 2026 | **Hosts:** India & Sri Lanka | **Champion:** India 🏆

2026 tournament data** scraped from ESPNcricinfo and Wikipedia.

---
**Pipeline:**
1. Load JSON & CSV data
2. Clean & preprocess each dataset
3. Merge datasets
4. Export clean CSVs & JSONs ready for Power BI

In [ ]:
import pandas as pd
import json
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded successfully')

: 

## 📋 1. Matches Summary — Load & Inspect

In [2]:
with open('WC_t20_json_files/t20_wc_match_results.json') as f:
    data = json.load(f)
data

{'matchSummary': [{'team1': 'Netherlands',
   'team2': 'Pakistan',
   'winner': 'Pakistan',
   'margin': '7 wickets',
   'ground': 'R. Premadasa Stadium, Colombo',
   'matchDate': 'Feb 07, 2026',
   'scorecard': 'T20I # 2800'},
  {'team1': 'West Indies',
   'team2': 'Scotland',
   'winner': 'West Indies',
   'margin': '120 runs',
   'ground': 'Wankhede Stadium, Mumbai',
   'matchDate': 'Feb 07, 2026',
   'scorecard': 'T20I # 2801'},
  {'team1': 'India',
   'team2': 'U.S.A.',
   'winner': 'India',
   'margin': '9 wickets',
   'ground': 'Wankhede Stadium, Mumbai',
   'matchDate': 'Feb 08, 2026',
   'scorecard': 'T20I # 2802'},
  {'team1': 'Afghanistan',
   'team2': 'New Zealand',
   'winner': 'New Zealand',
   'margin': '10 wickets',
   'ground': 'M.A. Chidambaram Stadium, Chennai',
   'matchDate': 'Feb 08, 2026',
   'scorecard': 'T20I # 2803'},
  {'team1': 'England',
   'team2': 'Nepal',
   'winner': 'England',
   'margin': '9 wickets',
   'ground': 'Wankhede Stadium, Mumbai',
   'match

In [3]:
# Convert match summary list to DataFrame
df_matches = pd.DataFrame(data['matchSummary'])
print(f'Shape: {df_matches.shape}')
df_matches.head()

Shape: (47, 7)


,team1,team2,winner,margin,ground,matchDate,scorecard
0,Netherlands,Pakistan,Pakistan,7 wickets,"R. Premadasa Stadium, Colombo","Feb 07, 2026",T20I # 2800
1,West Indies,Scotland,West Indies,120 runs,"Wankhede Stadium, Mumbai","Feb 07, 2026",T20I # 2801
2,India,U.S.A.,India,9 wickets,"Wankhede Stadium, Mumbai","Feb 08, 2026",T20I # 2802
3,Afghanistan,New Zealand,New Zealand,10 wickets,"M.A. Chidambaram Stadium, Chennai","Feb 08, 2026",T20I # 2803
4,England,Nepal,England,9 wickets,"Wankhede Stadium, Mumbai","Feb 08, 2026",T20I # 2804


### 🧹 1.1 Clean Match Data

In [4]:
# Parse matchDate to datetime
df_matches['matchDate'] = pd.to_datetime(df_matches['matchDate'], format='%b %d, %Y')

# Extract stage from ground + context (already a column in CSV version)
# Standardize margin — replace 'N/R' with NaN
df_matches['margin'] = df_matches['margin'].replace('N/R', np.nan)

# Flag No Result matches
df_matches['no_result'] = df_matches['winner'] == 'No Result'

# Extract match number from scorecard field
df_matches['match_number'] = df_matches['scorecard'].str.extract(r'(\d+)').astype(int)

print('Null values:')
print(df_matches.isnull().sum())
df_matches.dtypes

Null values:
team1           0
team2           0
winner          0
margin          1
ground          0
matchDate       0
scorecard       0
no_result       0
match_number    0
dtype: int64


team1                      str
team2                      str
winner                     str
margin                     str
ground                     str
matchDate       datetime64[us]
scorecard                  str
no_result                 bool
match_number             int64
dtype: object

In [5]:
# Check for duplicate matches
dupes = df_matches.duplicated(subset=['team1','team2','matchDate'])
print(f'Duplicate match rows: {dupes.sum()}')

# Unique winners
print('\nUnique winning teams:')
print(sorted(df_matches['winner'].unique()))

Duplicate match rows: 0

Unique winning teams:
['Australia', 'Canada', 'England', 'India', 'Ireland', 'Italy', 'Netherlands', 'New Zealand', 'No Result', 'Pakistan', 'South Africa', 'Sri Lanka', 'U.S.A.', 'West Indies', 'Zimbabwe']


In [6]:
# Wins per team (excluding No Result)
wins = df_matches[df_matches['winner'] != 'No Result']['winner'].value_counts()
print('Wins per team (Tournament):')
print(wins)

Wins per team (Tournament):
winner
India           8
New Zealand     6
England         6
South Africa    6
Pakistan        4
West Indies     3
Sri Lanka       3
Zimbabwe        3
Italy           2
Netherlands     1
Australia       1
Canada          1
U.S.A.          1
Ireland         1
Name: count, dtype: int64


## 🏏 2. Batting Summary — Load, Clean & Transform

In [7]:
df_bat = pd.read_csv('WC_t20_csv_files/Batting_Summary.csv')
print(f'Shape: {df_bat.shape}')
df_bat.head(10)

Shape: (30, 11)


,name,team,innings,runs,avg,sr,hundreds,fifties,fours,sixes,highest_score
0,Sahibzada Farhan,Pakistan,6,383,76.60,160.25,2,1,28,22,110
1,Tim Seifert,New Zealand,8,326,46.57,166.32,0,4,24,18,82
2,Sanju Samson,India,5,321,80.25,199.37,0,3,22,26,89
3,Ishan Kishan,India,7,298,49.66,170.28,0,3,20,20,76
4,Finn Allen,New Zealand,8,289,36.12,200.00,1,1,18,28,100
5,Brian Bennett,Zimbabwe,6,292,146.00,165.53,0,3,22,14,79
6,Shimron Hetmyer,West Indies,5,219,54.75,192.10,0,2,14,20,85
7,Harry Brook,England,7,210,35.00,170.73,1,0,16,12,100
8,Pathum Nissanka,Sri Lanka,5,208,52.00,158.77,1,1,18,8,100
9,Yuvraj Samra,Canada,4,175,43.75,168.26,1,0,10,14,110


In [8]:
# Check data types and nulls
print(df_bat.dtypes)
print('\nNull values:')
print(df_bat.isnull().sum())

name                 str
team                 str
innings            int64
runs               int64
avg              float64
sr               float64
hundreds           int64
fifties            int64
fours              int64
sixes              int64
highest_score      int64
dtype: object

Null values:
name             0
team             0
innings          0
runs             0
avg              0
sr               0
hundreds         0
fifties          0
fours            0
sixes            0
highest_score    0
dtype: int64


In [9]:
# ---- Cleaning ----
# Ensure numeric columns are correct dtype
numeric_cols = ['innings', 'runs', 'avg', 'sr', 'hundreds', 'fifties', 'fours', 'sixes']
for col in numeric_cols:
    df_bat[col] = pd.to_numeric(df_bat[col], errors='coerce')

# Fill NaN in avg with 0 (players who were not out in all innings)
df_bat['avg'] = df_bat['avg'].fillna(0)

# ---- Derived Columns ----
# Boundary percentage: what % of runs came from boundaries
df_bat['boundary_runs'] = (df_bat['fours'] * 4) + (df_bat['sixes'] * 6)
df_bat['boundary_pct'] = ((df_bat['boundary_runs'] / df_bat['runs']) * 100).round(2)

# Tag players with centuries
df_bat['has_century'] = df_bat['hundreds'] > 0

print('Batting DataFrame after cleaning:')
df_bat.head()

Batting DataFrame after cleaning:


,name,team,innings,runs,avg,sr,hundreds,fifties,fours,sixes,highest_score,boundary_runs,boundary_pct,has_century
0,Sahibzada Farhan,Pakistan,6,383,76.60,160.25,2,1,28,22,110,244,63.71,True
1,Tim Seifert,New Zealand,8,326,46.57,166.32,0,4,24,18,82,204,62.58,False
2,Sanju Samson,India,5,321,80.25,199.37,0,3,22,26,89,244,76.01,False
3,Ishan Kishan,India,7,298,49.66,170.28,0,3,20,20,76,200,67.11,False
4,Finn Allen,New Zealand,8,289,36.12,200.00,1,1,18,28,100,240,83.04,True


In [10]:
# ---- Key Aggregations ----

# Top 10 run scorers
print('🏏 Top 10 Run Scorers — T20 WC 2026:')
print(df_bat[['name','team','innings','runs','avg','sr']].sort_values('runs', ascending=False).head(10).to_string(index=False))

🏏 Top 10 Run Scorers — T20 WC 2026:
            name        team  innings  runs    avg     sr
Sahibzada Farhan    Pakistan        6   383  76.60 160.25
     Tim Seifert New Zealand        8   326  46.57 166.32
    Sanju Samson       India        5   321  80.25 199.37
    Ishan Kishan       India        7   298  49.66 170.28
   Brian Bennett    Zimbabwe        6   292 146.00 165.53
      Finn Allen New Zealand        8   289  36.12 200.00
 Shimron Hetmyer West Indies        5   219  54.75 192.10
     Harry Brook     England        7   210  35.00 170.73
 Pathum Nissanka   Sri Lanka        5   208  52.00 158.77
    Yuvraj Samra      Canada        4   175  43.75 168.26


In [11]:
# Top 10 by batting average (min 3 innings)
print('📊 Top 10 Best Batting Averages (min 3 innings):')
print(df_bat[df_bat['innings'] >= 3][['name','team','innings','runs','avg','sr']]
      .sort_values('avg', ascending=False).head(10).to_string(index=False))

📊 Top 10 Best Batting Averages (min 3 innings):
            name         team  innings  runs    avg     sr
   Brian Bennett     Zimbabwe        6   292 146.00 165.53
    Sanju Samson        India        5   321  80.25 199.37
Sahibzada Farhan     Pakistan        6   383  76.60 160.25
    David Miller South Africa        5   146  73.00 172.94
 Shimron Hetmyer  West Indies        5   219  54.75 192.10
 Pathum Nissanka    Sri Lanka        5   208  52.00 158.77
    Ishan Kishan        India        7   298  49.66 170.28
     Tim Seifert  New Zealand        8   326  46.57 166.32
    Yuvraj Samra       Canada        4   175  43.75 168.26
   Sikandar Raza     Zimbabwe        6   168  42.00 155.55


In [12]:
# Top 10 by strike rate (min 3 innings)
print('⚡ Top 10 Strike Rates (min 3 innings):')
print(df_bat[df_bat['innings'] >= 3][['name','team','innings','runs','avg','sr']]
      .sort_values('sr', ascending=False).head(10).to_string(index=False))

⚡ Top 10 Strike Rates (min 3 innings):
            name         team  innings  runs   avg     sr
      Finn Allen  New Zealand        8   289 36.12 200.00
    Sanju Samson        India        5   321 80.25 199.37
 Abhishek Sharma        India        7   163 27.16 192.94
 Shimron Hetmyer  West Indies        5   219 54.75 192.10
Suryakumar Yadav        India        7   165 33.00 188.50
 Johnson Charles  West Indies        5    95 23.75 188.11
   Rilee Rossouw South Africa        5   112 37.33 186.66
   Rezan Hossain        Italy        3   124 41.33 177.14
Brandon McMullen     Scotland        4   140 35.00 175.00
     Tilak Varma        India        6    98 24.50 175.00


In [13]:
# Most sixes
print('💥 Top 10 Six-Hitters:')
print(df_bat[['name','team','sixes','fours']].sort_values('sixes', ascending=False).head(10).to_string(index=False))

💥 Top 10 Six-Hitters:
            name        team  sixes  fours
      Finn Allen New Zealand     28     18
    Sanju Samson       India     26     22
Sahibzada Farhan    Pakistan     22     28
    Ishan Kishan       India     20     20
 Shimron Hetmyer West Indies     20     14
     Tim Seifert New Zealand     18     24
Suryakumar Yadav       India     18     10
 Abhishek Sharma       India     16     14
   Brian Bennett    Zimbabwe     14     22
    Yuvraj Samra      Canada     14     10


In [14]:
# Players with centuries
print('💯 Century scorers in T20 WC 2026:')
print(df_bat[df_bat['hundreds'] > 0][['name','team','hundreds','runs','avg','sr','highest_score']]
      .sort_values('hundreds', ascending=False).to_string(index=False))

💯 Century scorers in T20 WC 2026:
            name        team  hundreds  runs   avg     sr  highest_score
Sahibzada Farhan    Pakistan         2   383 76.60 160.25            110
      Finn Allen New Zealand         1   289 36.12 200.00            100
     Harry Brook     England         1   210 35.00 170.73            100
 Pathum Nissanka   Sri Lanka         1   208 52.00 158.77            100
    Yuvraj Samra      Canada         1   175 43.75 168.26            110
   Jacob Bethell     England         1   155 31.00 165.95            101


In [15]:
# Team-wise batting totals
team_bat = df_bat.groupby('team').agg(
    total_runs=('runs', 'sum'),
    avg_sr=('sr', 'mean'),
    total_sixes=('sixes', 'sum'),
    total_fours=('fours', 'sum'),
    centuries=('hundreds', 'sum'),
    fifties=('fifties', 'sum')
).sort_values('total_runs', ascending=False).round(2)

print('\n🌍 Team-wise Batting Aggregates:')
print(team_bat.to_string())


🌍 Team-wise Batting Aggregates:
              total_runs  avg_sr  total_sixes  total_fours  centuries  fifties
team                                                                          
India               1045  185.22           90           72          0        9
New Zealand          970  169.59           68           70          1        8
Pakistan             521  150.53           26           40          2        2
England              499  156.73           28           38          2        1
Zimbabwe             460  160.54           24           34          0        5
West Indies          424  161.44           34           30          0        3
South Africa         389  174.45           32           26          0        4
Sri Lanka            208  158.77            8           18          1        1
Canada               175  168.26           14           10          1        0
Scotland             140  175.00           10           10          0        1
Italy              

## 🎳 3. Bowling Summary — Load, Clean & Transform

In [16]:
df_bowl = pd.read_csv('WC_t20_csv_files/Bowling_Summary.csv')
print(f'Shape: {df_bowl.shape}')
df_bowl.head(10)

Shape: (25, 10)


,name,team,innings,wickets,avg,economy,sr,best_bowling,four_wickets,five_wickets
0,Jasprit Bumrah,India,8,14,12.42,6.21,12.00,4/15,1,0
1,Varun Chakravarthy,India,9,14,20.50,8.20,15.00,4/22,1,0
2,Shadley van Schalkwyk,USA,4,13,7.76,6.80,6.84,4/22,2,0
3,Blessing Muzarabani,Zimbabwe,6,13,10.27,7.88,7.80,4/28,1,0
4,Adil Rashid,England,8,13,16.84,7.23,13.96,3/16,0,0
5,Arshdeep Singh,India,8,11,18.27,7.89,13.90,3/18,0,0
6,Mark Wood,England,7,10,17.40,9.80,10.64,3/22,0,0
7,Rehan Ahmed,England,7,9,19.44,7.20,16.22,3/18,0,0
8,Romario Shepherd,West Indies,5,9,14.55,9.40,9.28,5/20,0,1
9,Wanindu Hasaranga,Sri Lanka,5,9,15.33,7.40,12.44,3/24,0,0


In [17]:
print(df_bowl.dtypes)
print('\nNull values:')
print(df_bowl.isnull().sum())

name                str
team                str
innings           int64
wickets           int64
avg             float64
economy         float64
sr              float64
best_bowling        str
four_wickets      int64
five_wickets      int64
dtype: object

Null values:
name            0
team            0
innings         0
wickets         0
avg             0
economy         0
sr              0
best_bowling    0
four_wickets    0
five_wickets    0
dtype: int64


In [18]:
# ---- Cleaning ----
numeric_bowl_cols = ['innings', 'wickets', 'avg', 'economy', 'sr', 'four_wickets', 'five_wickets']
for col in numeric_bowl_cols:
    df_bowl[col] = pd.to_numeric(df_bowl[col], errors='coerce')

# Handle infinity values (e.g. bowlers with 0 wickets)
df_bowl['avg'] = df_bowl['avg'].replace([np.inf, -np.inf], np.nan)
df_bowl['sr'] = df_bowl['sr'].replace([np.inf, -np.inf], np.nan)

# Parse best bowling: split into wickets and runs
df_bowl[['bb_wickets', 'bb_runs']] = df_bowl['best_bowling'].str.split('/', expand=True).astype(float)

print('Bowling DataFrame after cleaning:')
df_bowl.head()

Bowling DataFrame after cleaning:


,name,team,innings,wickets,avg,economy,sr,best_bowling,four_wickets,five_wickets,bb_wickets,bb_runs
0,Jasprit Bumrah,India,8,14,12.42,6.21,12.00,4/15,1,0,4.0,15.0
1,Varun Chakravarthy,India,9,14,20.50,8.20,15.00,4/22,1,0,4.0,22.0
2,Shadley van Schalkwyk,USA,4,13,7.76,6.80,6.84,4/22,2,0,4.0,22.0
3,Blessing Muzarabani,Zimbabwe,6,13,10.27,7.88,7.80,4/28,1,0,4.0,28.0
4,Adil Rashid,England,8,13,16.84,7.23,13.96,3/16,0,0,3.0,16.0


In [19]:
# Top wicket takers
print('🎳 Top 10 Wicket Takers — T20 WC 2026:')
print(df_bowl[['name','team','innings','wickets','avg','economy','sr','best_bowling']]
      .sort_values('wickets', ascending=False).head(10).to_string(index=False))

🎳 Top 10 Wicket Takers — T20 WC 2026:
                 name        team  innings  wickets   avg  economy    sr best_bowling
       Jasprit Bumrah       India        8       14 12.42     6.21 12.00         4/15
   Varun Chakravarthy       India        9       14 20.50     8.20 15.00         4/22
Shadley van Schalkwyk         USA        4       13  7.76     6.80  6.84         4/22
  Blessing Muzarabani    Zimbabwe        6       13 10.27     7.88  7.80         4/28
          Adil Rashid     England        8       13 16.84     7.23 13.96         3/16
       Arshdeep Singh       India        8       11 18.27     7.89 13.90         3/18
            Mark Wood     England        7       10 17.40     9.80 10.64         3/22
          Rehan Ahmed     England        7        9 19.44     7.20 16.22         3/18
     Romario Shepherd West Indies        5        9 14.55     9.40  9.28         5/20
    Wanindu Hasaranga   Sri Lanka        5        9 15.33     7.40 12.44         3/24


In [20]:
# Best economy (min 4 innings)
print('💨 Top 10 Most Economical Bowlers (min 4 innings):')
print(df_bowl[df_bowl['innings'] >= 4][['name','team','innings','wickets','economy','avg']]
      .sort_values('economy').head(10).to_string(index=False))

💨 Top 10 Most Economical Bowlers (min 4 innings):
                 name        team  innings  wickets  economy   avg
       Jasprit Bumrah       India        8       14     6.21 12.42
Shadley van Schalkwyk         USA        4       13     6.80  7.76
          Rashid Khan Afghanistan        4        8     6.80 14.25
           Axar Patel       India        7        7     6.85 19.71
     Mitchell Santner New Zealand        8        8     7.10 22.75
          Rehan Ahmed     England        7        9     7.20 19.44
       Gudakesh Motie West Indies        5        6     7.20 16.33
          Adil Rashid     England        8       13     7.23 16.84
        Sikandar Raza    Zimbabwe        6        7     7.32 21.28
    Wanindu Hasaranga   Sri Lanka        5        9     7.40 15.33


In [21]:
# 5-wicket hauls
print('⭐ Bowlers with 5-Wicket Hauls:')
print(df_bowl[df_bowl['five_wickets'] > 0][['name','team','wickets','best_bowling','economy']].to_string(index=False))

⭐ Bowlers with 5-Wicket Hauls:
            name        team  wickets best_bowling  economy
Romario Shepherd West Indies        9         5/20     9.40
 Junaid Siddique         UAE        7         5/35     7.42


In [22]:
# 4-wicket hauls
print('⭐ Bowlers with 4-Wicket Hauls:')
print(df_bowl[df_bowl['four_wickets'] > 0][['name','team','wickets','four_wickets','best_bowling','economy']].to_string(index=False))

⭐ Bowlers with 4-Wicket Hauls:
                 name        team  wickets  four_wickets best_bowling  economy
       Jasprit Bumrah       India       14             1         4/15     6.21
   Varun Chakravarthy       India       14             1         4/22     8.20
Shadley van Schalkwyk         USA       13             2         4/22     6.80
  Blessing Muzarabani    Zimbabwe       13             1         4/28     7.88
       Gudakesh Motie West Indies        6             1         4/18     7.20


In [23]:
# Team bowling stats
team_bowl = df_bowl.groupby('team').agg(
    total_wickets=('wickets', 'sum'),
    avg_economy=('economy', 'mean'),
    fifers=('five_wickets', 'sum'),
    four_fers=('four_wickets', 'sum')
).sort_values('total_wickets', ascending=False).round(2)

print('\n🌍 Team-wise Bowling Aggregates:')
print(team_bowl.to_string())


🌍 Team-wise Bowling Aggregates:
              total_wickets  avg_economy  fifers  four_fers
team                                                       
India                    54         7.65       0          2
England                  32         8.08       0          0
South Africa             23         8.50       0          0
Zimbabwe                 20         7.60       0          1
West Indies              15         8.30       1          1
New Zealand              14         5.53       0          0
USA                      13         6.80       0          2
Pakistan                 11         9.65       0          0
Sri Lanka                 9         7.40       0          0
Afghanistan               8         6.80       0          0
UAE                       7         7.42       1          0
Bangladesh                6         8.40       0          0


## 👤 4. Players Information — Load & Clean

In [24]:
df_players = pd.read_csv('WC_t20_csv_files/Players_Information.csv')
print(f'Shape: {df_players.shape}')
df_players.head(10)

Shape: (40, 6)


,name,team,role,batting_style,bowling_style,age
0,Sahibzada Farhan,Pakistan,Batter,Right-hand bat,Right-arm off-break,30
1,Tim Seifert,New Zealand,Wicketkeeper-Batter,Right-hand bat,NaN,29
2,Sanju Samson,India,Wicketkeeper-Batter,Right-hand bat,NaN,31
3,Ishan Kishan,India,Wicketkeeper-Batter,Left-hand bat,NaN,26
4,Finn Allen,New Zealand,Batter,Right-hand bat,Right-arm off-break,27
5,Brian Bennett,Zimbabwe,Batter,Right-hand bat,Right-arm medium,22
6,Shimron Hetmyer,West Indies,Batter,Left-hand bat,NaN,27
7,Harry Brook,England,Batter,Right-hand bat,Right-arm medium,27
8,Pathum Nissanka,Sri Lanka,Batter,Right-hand bat,Right-arm off-break,27
9,Yuvraj Samra,Canada,Batter,Right-hand bat,Right-arm off-break,19


In [25]:
# Check for nulls and types
print(df_players.isnull().sum())
print(df_players['role'].value_counts())

name             0
team             0
role             0
batting_style    0
bowling_style    6
age              0
dtype: int64
role
All-rounder            14
Batter                 13
Bowler                  9
Wicketkeeper-Batter     4
Name: count, dtype: int64


In [26]:
# Clean bowlling_style: replace 'NA' with np.nan
df_players['bowling_style'] = df_players['bowling_style'].replace('NA', np.nan)

# Categorize batting hand
df_players['batting_hand'] = df_players['batting_style'].str.extract(r'^(Right|Left)')

print('Players by role:')
print(df_players['role'].value_counts())
print('\nPlayers by batting hand:')
print(df_players['batting_hand'].value_counts())

Players by role:


role
All-rounder            14
Batter                 13
Bowler                  9
Wicketkeeper-Batter     4
Name: count, dtype: int64

Players by batting hand:
batting_hand
Right    29
Left     11
Name: count, dtype: int64


## 🔗 5. Merge Batting + Player Info

In [27]:
df_bat_full = df_bat.merge(df_players[['name','role','batting_style','bowling_style','batting_hand','age']], 
                            on='name', how='left')
print(f'Merged batting shape: {df_bat_full.shape}')
df_bat_full.head()

Merged batting shape: (30, 19)


,name,team,innings,runs,avg,sr,hundreds,fifties,fours,sixes,highest_score,boundary_runs,boundary_pct,has_century,role,batting_style,bowling_style,batting_hand,age
0,Sahibzada Farhan,Pakistan,6,383,76.60,160.25,2,1,28,22,110,244,63.71,True,Batter,Right-hand bat,Right-arm off-break,Right,30.0
1,Tim Seifert,New Zealand,8,326,46.57,166.32,0,4,24,18,82,204,62.58,False,Wicketkeeper-Batter,Right-hand bat,NaN,Right,29.0
2,Sanju Samson,India,5,321,80.25,199.37,0,3,22,26,89,244,76.01,False,Wicketkeeper-Batter,Right-hand bat,NaN,Right,31.0
3,Ishan Kishan,India,7,298,49.66,170.28,0,3,20,20,76,200,67.11,False,Wicketkeeper-Batter,Left-hand bat,NaN,Left,26.0
4,Finn Allen,New Zealand,8,289,36.12,200.00,1,1,18,28,100,240,83.04,True,Batter,Right-hand bat,Right-arm off-break,Right,27.0


In [28]:
# Left-hand vs Right-hand run comparison
print('Runs by batting hand (top scorers):')
print(df_bat_full.groupby('batting_hand')['runs'].agg(['sum','mean','count']).round(2))

Runs by batting hand (top scorers):
               sum    mean  count
batting_hand                     
Left          1318  164.75      8
Right         3216  178.67     18


In [29]:
# Runs by player role
print('\nRuns by player role:')
print(df_bat_full.groupby('role')['runs'].agg(['sum','mean','count']).sort_values('sum', ascending=False).round(2))


Runs by player role:


                      sum    mean  count
role                                    
Batter               2558  196.77     13
Wicketkeeper-Batter  1076  269.00      4
All-rounder           860  122.86      7
Bowler                 40   20.00      2


## 🔗 6. Merge Bowling + Player Info

In [30]:
df_bowl_full = df_bowl.merge(df_players[['name','role','bowling_style','age']], 
                              on='name', how='left')
print(f'Merged bowling shape: {df_bowl_full.shape}')
df_bowl_full.head()

Merged bowling shape: (25, 15)


,name,team,innings,wickets,avg,economy,sr,best_bowling,four_wickets,five_wickets,bb_wickets,bb_runs,role,bowling_style,age
0,Jasprit Bumrah,India,8,14,12.42,6.21,12.00,4/15,1,0,4.0,15.0,Bowler,Right-arm fast,30.0
1,Varun Chakravarthy,India,9,14,20.50,8.20,15.00,4/22,1,0,4.0,22.0,Bowler,Right-arm leg-break,33.0
2,Shadley van Schalkwyk,USA,4,13,7.76,6.80,6.84,4/22,2,0,4.0,22.0,Bowler,Right-arm medium,30.0
3,Blessing Muzarabani,Zimbabwe,6,13,10.27,7.88,7.80,4/28,1,0,4.0,28.0,Bowler,Right-arm fast,26.0
4,Adil Rashid,England,8,13,16.84,7.23,13.96,3/16,0,0,3.0,16.0,Bowler,Right-arm leg-break,36.0


In [31]:
# Wickets by bowling style
print('Wickets by bowling style:')
print(df_bowl_full.groupby('bowling_style')['wickets'].sum().sort_values(ascending=False))

Wickets by bowling style:
bowling_style
Right-arm leg-break      53
Right-arm fast           45
Left-arm orthodox        21
Left-arm fast-medium     19
Right-arm medium         13
Right-arm fast-medium     9
Right-arm medium-fast     8
Right-arm off-break       7
Name: wickets, dtype: int64


## 📊 7. Match Analysis

In [32]:
df_total = pd.read_csv('WC_t20_csv_files/Total_Matches.csv')
print(f'Total matches: {len(df_total)}')
df_total.head()

Total matches: 51


,match_id,team1,team2,winner,margin,margin_type,ground,match_date,stage,scorecard
0,1,Netherlands,Pakistan,Pakistan,7,wickets,R. Premadasa Stadium Colombo,Feb 07 2026,Group A,T20I # 2800
1,2,West Indies,Scotland,West Indies,120,runs,Wankhede Stadium Mumbai,Feb 07 2026,Group C,T20I # 2801
2,3,India,USA,India,9,wickets,Wankhede Stadium Mumbai,Feb 08 2026,Group A,T20I # 2802
3,4,Afghanistan,New Zealand,New Zealand,10,wickets,MA Chidambaram Stadium Chennai,Feb 08 2026,Group D,T20I # 2803
4,5,England,Nepal,England,9,wickets,Wankhede Stadium Mumbai,Feb 08 2026,Group C,T20I # 2804


In [33]:
# Clean Total_Matches CSV
df_total['match_date'] = pd.to_datetime(df_total['match_date'], format='%b %d %Y')

# Stage breakdown
print('Matches by stage:')
print(df_total['stage'].value_counts())

Matches by stage:
stage
Group A            9
Group C            9
Group D            9
Group B            9
Super 8 Group 2    6
Super 8 Group 1    6
Semi-Final 1       1
Semi-Final 2       1
Final              1
Name: count, dtype: int64


In [34]:
# Win type distribution (by wickets vs by runs)
win_type = df_total['margin_type'].value_counts()
print('\nWin type distribution:')
print(win_type)


Win type distribution:
margin_type
wickets       35
runs          14
super over     1
NR             1
Name: count, dtype: int64


In [35]:
# Matches per venue
print('\nMatches per venue:')
print(df_total['ground'].value_counts().head(10))


Matches per venue:


ground
R. Premadasa Stadium Colombo                     10
Wankhede Stadium Mumbai                           9
MA Chidambaram Stadium Chennai                    9
Narendra Modi Stadium Ahmedabad                   8
Pallekele International Cricket Stadium Kandy     7
Arun Jaitley Stadium Delhi                        6
Eden Gardens Kolkata                              2
Name: count, dtype: int64


In [36]:
# All India matches
india_matches = df_total[(df_total['team1'] == 'India') | (df_total['team2'] == 'India')]
print(f"India played {len(india_matches)} matches")
india_wins = india_matches[india_matches['winner'] == 'India']
print(f"India won {len(india_wins)} matches")
print(india_matches[['team1','team2','winner','margin','margin_type','ground','stage']].to_string(index=False))

India played 9 matches
India won 8 matches
      team1        team2       winner margin margin_type                                        ground           stage
      India          USA        India      9     wickets                       Wankhede Stadium Mumbai         Group A
      India      Namibia        India    120        runs                    Arun Jaitley Stadium Delhi         Group A
      India     Pakistan        India      6     wickets Pallekele International Cricket Stadium Kandy         Group A
Netherlands        India        India      8     wickets               Narendra Modi Stadium Ahmedabad         Group A
      India South Africa South Africa     76        runs               Narendra Modi Stadium Ahmedabad Super 8 Group 1
      India     Zimbabwe        India     72        runs                MA Chidambaram Stadium Chennai Super 8 Group 1
      India  West Indies        India      5     wickets                          Eden Gardens Kolkata Super 8 Group 1
    E

## 💾 8. Export Clean Data — Ready for Power BI

In [37]:
# Export clean CSVs
df_bat_full.to_csv('WC_t20_csv_files/Batting_Summary_clean.csv', index=False)
df_bowl_full.to_csv('WC_t20_csv_files/Bowling_Summary_clean.csv', index=False)
df_players.to_csv('WC_t20_csv_files/Players_Information_clean.csv', index=False)
df_total.to_csv('WC_t20_csv_files/Total_Matches_clean.csv', index=False)
df_matches.to_csv('WC_t20_csv_files/Match_Results_clean.csv', index=False)

# Export team aggregates
team_bat.to_csv('WC_t20_csv_files/Team_Batting_Aggregates.csv')
team_bowl.to_csv('WC_t20_csv_files/Team_Bowling_Aggregates.csv')

print('✅ All clean datasets exported to WC_t20_csv_files/')
print('\nFiles ready for Power BI:')
import os
for f in sorted(os.listdir('WC_t20_csv_files/')):
    size = os.path.getsize(f'WC_t20_csv_files/{f}')
    print(f'  📄 {f} ({size:,} bytes)')

✅ All clean datasets exported to WC_t20_csv_files/

Files ready for Power BI:
  📄 Batting_Summary.csv (1,721 bytes)
  📄 Batting_Summary_clean.csv (3,568 bytes)
  📄 Bowling_Summary.csv (1,414 bytes)
  📄 Bowling_Summary_clean.csv (2,308 bytes)
  📄 Match_Results_clean.csv (4,938 bytes)
  📄 Players_Information.csv (2,766 bytes)
  📄 Players_Information_clean.csv (2,996 bytes)
  📄 Team_Batting_Aggregates.csv (415 bytes)
  📄 Team_Bowling_Aggregates.csv (291 bytes)
  📄 Total_Matches.csv (5,418 bytes)
  📄 Total_Matches_clean.csv (5,367 bytes)


## 🏆 9. Tournament Summary — Key Stats
A final summary of the 2026 T20 World Cup

In [38]:
print('=' * 60)
print('   🏏 ICC Men\'s T20 World Cup 2026 — Summary Stats')
print('=' * 60)
print(f"🏆 Champion         : India (3rd title, 2nd consecutive)")
print(f"🥈 Runner-up        : New Zealand")
print(f"🏟️  Final venue       : Narendra Modi Stadium, Ahmedabad")
print(f"📅 Tournament dates : Feb 07 – Mar 08, 2026")
print(f"🌍 Co-hosts         : India & Sri Lanka")
print(f"⚽ Total matches    : 55 (across 8 venues)")
print()
print('🏏 Batting Records:')
top_bat = df_bat.sort_values('runs', ascending=False).iloc[0]
print(f"   Most runs        : {top_bat['name']} ({top_bat['team']}) — {top_bat['runs']} runs")
print(f"   Most centuries   : Sahibzada Farhan (PAK) — 2 hundreds")
print(f"   Fastest century  : Finn Allen (NZ) — 33 balls vs South Africa (SF)")
print(f"   Highest score    : Sahibzada Farhan — 110 vs Namibia")
print()
print('🎳 Bowling Records:')
top_bowl = df_bowl.sort_values('wickets', ascending=False).iloc[0]
print(f"   Most wickets     : {top_bowl['name']} ({top_bowl['team']}) — {top_bowl['wickets']} wickets")
print(f"   Best figures     : Romario Shepherd (WI) — 5/20 vs Scotland")
print(f"   Best economy     : Jasprit Bumrah (IND) — 6.21")
print()
print('🎖️  Awards:')
print(f"   Player of Tournament : Sanju Samson (India)")
print(f"   Player of Final      : Jasprit Bumrah (India) — 4/15")
print('=' * 60)


   🏏 ICC Men's T20 World Cup 2026 — Summary Stats
🏆 Champion         : India (3rd title, 2nd consecutive)
🥈 Runner-up        : New Zealand
🏟️  Final venue       : Narendra Modi Stadium, Ahmedabad
📅 Tournament dates : Feb 07 – Mar 08, 2026
🌍 Co-hosts         : India & Sri Lanka
⚽ Total matches    : 55 (across 8 venues)

🏏 Batting Records:
   Most runs        : Sahibzada Farhan (Pakistan) — 383 runs
   Most centuries   : Sahibzada Farhan (PAK) — 2 hundreds
   Fastest century  : Finn Allen (NZ) — 33 balls vs South Africa (SF)
   Highest score    : Sahibzada Farhan — 110 vs Namibia

🎳 Bowling Records:
   Most wickets     : Jasprit Bumrah (India) — 14 wickets
   Best figures     : Romario Shepherd (WI) — 5/20 vs Scotland
   Best economy     : Jasprit Bumrah (IND) — 6.21

🎖️  Awards:
   Player of Tournament : Sanju Samson (India)
   Player of Final      : Jasprit Bumrah (India) — 4/15


---
## ✅ Data Preprocessing Complete!

**Clean datasets ready for Power BI:**
- `Batting_Summary_clean.csv` — enriched with role, batting hand, boundary %
- `Bowling_Summary_clean.csv` — enriched with role, bowling style, best bowling split
- `Players_Information_clean.csv` — full player profiles
- `Total_Matches_clean.csv` — all 55 matches with dates
- `Match_Results_clean.csv` — from JSON, parsed dates
- `Team_Batting_Aggregates.csv` — team-level batting stats
- `Team_Bowling_Aggregates.csv` — team-level bowling stats

**Next step:** Import into Power BI → create relationships → build dashboards with DAX measures.